# Cooling Tower Detection Pipeline

This notebook demonstrates the YOLO-based detection pipeline for cooling towers in aerial imagery.

## 1. Setup and Imports

In [ ]:
import sys
sys.path.insert(0, '..')

from src.detection import CoolingTowerDetector
from src.config_loader import Config
from src.utils import visualize_detections
import matplotlib.pyplot as plt
from pathlib import Path

print("✓ Imports successful")

## 2. Load Configuration

In [ ]:
config = Config("../config/config.yaml")
print(f"Input directory: {config.get('paths.input_dir')}")
print(f"Output directory: {config.get('paths.output_dir')}")
print(f"Confidence threshold: {config.get('detection.confidence_threshold')}")

## 3. Initialize Detector

In [ ]:
detector = CoolingTowerDetector(
    model_path=config.get('detection.model_path'),
    conf_threshold=config.get('detection.confidence_threshold'),
    img_size=config.get('detection.image_size'),
    augment=config.get('detection.augment')
)

## 4. Run Detection on Sample Images

In [ ]:
# Process directory
results = detector.process_directory(
    input_dir=config.get('paths.input_dir'),
    output_file=f"{config.get('paths.detections_dir')}/detections.pkl",
    max_workers=config.get('processing.max_workers')
)

print(f"\nProcessed {len(results)} images with detections")

## 5. Visualize Results

In [ ]:
# Visualize first 5 results
for i, (img_path, boxes, confs) in enumerate(results[:5]):
    print(f"\nImage {i+1}: {Path(img_path).name}")
    print(f"  Detections: {len(boxes)}")
    
    visualize_detections(
        img_path,
        boxes.cpu().numpy(),
        confs.cpu().numpy(),
        show=True
    )

## 6. Summary Statistics

In [ ]:
import numpy as np

# Calculate statistics
num_detections = [len(boxes) for _, boxes, _ in results]
total_detections = sum(num_detections)

print("Detection Statistics:")
print(f"  Total images: {len(results)}")
print(f"  Total detections: {total_detections}")
print(f"  Mean detections per image: {np.mean(num_detections):.2f}")
print(f"  Median detections per image: {np.median(num_detections):.0f}")
print(f"  Max detections in single image: {np.max(num_detections)}")

# Plot histogram
plt.figure(figsize=(10, 6))
plt.hist(num_detections, bins=20, edgecolor='black')
plt.xlabel('Number of Detections per Image')
plt.ylabel('Frequency')
plt.title('Distribution of Detections')
plt.grid(axis='y', alpha=0.3)
plt.show()

## Next Steps

1. Review detections using `02_hitl_review.ipynb`
2. Run segmentation on approved detections using `03_segmentation_pipeline.ipynb`